# Automated Data Preprocessing Pipeline

This notebook is a **thin interactive layer** over the `data_prep` package in `src/`.
The logic (loading, cleaning, exporting, charts, profiling) lives in importable,
tested Python modules, so the same steps can be reused in a script or a future CLI.

What it does:

1. Loads CSV, Excel, JSON/JSON Lines or Parquet files (with real encoding detection)
2. Cleans them (missing values, duplicates, safe column names) and prints exactly what changed
3. Saves the cleaned data, draws bar/pie charts and builds an HTML profiling report

Run the cells top to bottom. Answers to the questions are validated: an empty answer
means "use the default shown in the prompt" and never causes an endless re-prompt.


## 1. Setup

If you have not installed the dependencies yet, run this in a terminal:

```bash
pip install -r requirements.txt
```

The cell below only imports things and configures logging. It does **not** prompt,
create folders or read data, so importing the package is always safe.


In [ ]:
import logging
import os
import sys
from datetime import datetime
from pathlib import Path

# Make src/ importable when running from a fresh clone (no install needed).
SRC = Path.cwd() / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_prep import (
    CleaningOptions,
    DataPrepError,
    EmptyOutputError,
    build_report,
    clean_data,
    list_data_files,
    load_table,
    make_charts,
    parse_column_selection,
    parse_file_selection,
    parse_sample_size,
    save_outputs,
)
from data_prep.interactive import PromptError

OUTPUT_FOLDER = Path("output")
OUTPUT_FOLDER.mkdir(exist_ok=True)

# force=True so re-running the cell does not silently keep the first log file.
logging.basicConfig(
    filename=str(OUTPUT_FOLDER / f"processing_{datetime.now():%Y%m%d_%H%M%S}.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
)

print(f"data_prep ready. Outputs go to: {OUTPUT_FOLDER.resolve()}")


## 2. Choose the folder and the datasets

Only supported files in the chosen folder are listed, sorted by name, and Excel
lock files (`~$book.xlsx`) are skipped. Press Enter to process the last file.


In [ ]:
def get_valid_directory() -> Path:
    """Ask for a folder until the answer is an existing directory."""
    while True:
        raw = input(
            "Enter dataset folder path [default: current directory]: "
        ).strip()
        path = Path(raw).expanduser() if raw else Path.cwd()
        if path.is_file():
            print(f"⚠️ '{path}' is a file, not a folder. Using its parent folder.")
            path = path.parent
        if path.is_dir():
            return path
        print(f"❌ '{path}' is not a valid directory. Please try again.")


DATA_FOLDER = get_valid_directory()
files = list_data_files(DATA_FOLDER)

if not files:
    raise SystemExit(f"❌ No datasets found in: {DATA_FOLDER}")

print("\nAvailable datasets:")
for number, path in enumerate(files, start=1):
    print(f"  {number}. {path.name}")

while True:
    answer = input(
        "\nEnter number(s) of dataset(s) to process, comma-separated "
        "(e.g. 1,3) [default: last]: "
    ).strip()
    try:
        SELECTED_FILES = [files[index] for index in parse_file_selection(answer, len(files))]
        break
    except PromptError as error:
        print(f"⚠️ {error}")

print("\nSelected: " + ", ".join(path.name for path in SELECTED_FILES))


## 3. Cleaning questions

Answers are parsed by tested helpers in `data_prep.interactive`, so a typo re-asks
instead of silently choosing something else.

Note: **text is never converted to numbers unless you ask for it.** ZIP codes,
IDs and phone numbers keep their leading zeros.


In [ ]:
def ask_cleaning_options(columns: list[str], row_count: int) -> tuple[CleaningOptions, dict]:
    """Collect cleaning/chart preferences and build CleaningOptions."""
    print("\n--- Data Cleaning Options ---")
    while True:
        choice = input(
            "Handle missing values?\n [1] Drop rows\n [2] Fill mean/mode\n "
            "[3] Do nothing\n Choice: "
        ).strip()
        if choice in {"1", "2", "3"}:
            break
        print("⚠️ Invalid choice. Please select 1, 2, or 3.")
    missing = {"1": "drop", "2": "fill", "3": "none"}[choice]

    while True:
        dedupe_answer = input("Drop duplicate rows? (y/n): ").strip().lower()
        if dedupe_answer in {"y", "n"}:
            break
        print("⚠️ Invalid choice. Please select y or n.")
    dedupe = dedupe_answer == "y"

    subset = None
    if dedupe:
        raw = input(
            "Optional: key columns that identify a duplicate, comma-separated "
            "(e.g. Booking ID). Press Enter to compare whole rows: "
        ).strip()
        subset = [name.strip() for name in raw.split(",") if name.strip()] or None

    convert = input(
        "Convert text columns to numbers where nothing would be lost? (y/n) "
        "[default: n]: "
    ).strip().lower() == "y"

    print("\n--- Visualization Options ---")
    print(f"Available columns: {columns}")
    bar_columns: list[str] = []
    pie_columns: list[str] = []
    if input("Manually select columns for visualizations? (y/n): ").strip().lower() == "y":
        raw_bar = input("Columns for bar charts (comma-separated) or 'none': ")
        bar_columns, unknown_bar = parse_column_selection(raw_bar, columns)
        if unknown_bar:
            print(f"⚠️ Not found in this dataset: {', '.join(unknown_bar)}")

        raw_pie = input("Columns for pie charts (comma-separated) or 'none': ")
        pie_columns, unknown_pie = parse_column_selection(raw_pie, columns)
        if unknown_pie:
            print(f"⚠️ Not found in this dataset: {', '.join(unknown_pie)}")

    while True:
        suggestion = 50000 if row_count > 150000 else None
        default_text = suggestion if suggestion is not None else "full dataset"
        raw = input(
            f"Enter sample size for visualizations (e.g. 50000) "
            f"[default: {default_text}]: "
        )
        try:
            sample_size = parse_sample_size(raw, row_count)
            break
        except PromptError as error:
            print(f"⚠️ {error}")

    return CleaningOptions(
        missing_values=missing,
        drop_duplicates=dedupe,
        duplicate_subset=subset,
        numeric_coercion="safe" if convert else "none",
    ), {
        "bar_columns": bar_columns,
        "pie_columns": pie_columns,
        "sample_size": sample_size,
    }


## 4. Run the pipeline

Each file is processed independently: a problem with one dataset is reported and the
next one still runs. Everything written into `output/` contains your raw data -
see the README's privacy note before committing or sharing those files.


In [ ]:
def process_file(path: Path) -> None:
    """Load, ask, clean, save, chart and profile one dataset."""
    base = path.stem
    print(f"\n{'=' * 20} Processing {base} {'=' * 20}")
    logging.info("Processing file: %s", base)

    try:
        raw_df = load_table(path)
    except DataPrepError as error:
        print(f"❌ Could not load {path.name}: {error}")
        logging.error("Load failed for %s: %s", path.name, error)
        return

    print(f"✅ Loaded with shape {raw_df.shape}")
    print(raw_df.head())

    options, plot_choices = ask_cleaning_options(list(raw_df.columns), len(raw_df))
    cleaned_df, summary = clean_data(raw_df, options)

    print("\n📋 What changed:")
    for line in summary.to_lines():
        print(f"  {line}")

    try:
        paths = save_outputs(cleaned_df, base, OUTPUT_FOLDER)
    except EmptyOutputError as error:
        print(f"❌ {error}")
        logging.error("Refused to save empty output for %s", base)
    else:
        print(f"✅ Saved CSV -> {paths.csv}")
        if paths.excel:
            print(f"✅ Saved Excel -> {paths.excel}")
        for note in paths.notes:
            print(f"⚠️ {note}")

    chart_paths, notes = make_charts(
        cleaned_df,
        base,
        OUTPUT_FOLDER,
        bar_columns=plot_choices["bar_columns"],
        pie_columns=plot_choices["pie_columns"],
        sample_size=plot_choices["sample_size"],
    )
    for chart in chart_paths:
        print(f"✅ Chart -> {chart}")
    for note in notes:
        print(f"⚠️ {note}")

    try:
        # The report describes the data as it arrived, so it can be compared
        # with the cleaned output.
        report_path = build_report(raw_df, base, OUTPUT_FOLDER)
        print(f"✅ Report -> {report_path}")
    except DataPrepError as error:
        print(f"❌ Report not generated: {error}")
        logging.error("Report failed for %s: %s", base, error)


for file_path in SELECTED_FILES:
    process_file(file_path)

print("\n🎉 Done.")
logging.info("Automation complete")


## 5. Next steps

* Run the tests: `python -m pytest -q` (see README for what they cover).
* Every run writes a log file and a `CleaningSummary` you can turn into a
  machine-readable data-quality report.

Known limitations are listed in the README (for example: only the first Excel
sheet is read, dates are not parsed automatically, and there is no CLI yet).
